In [ ]:
import glob
from pathlib import Path

import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

DATA_DIR = Path('../data')

In [ ]:
# from osf import sync_osf
# csv_files = sync_osf(DATA_DIR / "raw")

In [ ]:
csv_files = glob.glob(f'{DATA_DIR}/raw/*.csv')

In [ ]:
sessions = [pd.read_csv(f) for f in csv_files]
kept_columns = ['random_id', 'chunk_length', 'chunk', 'rt', 'round_index', 'bot_move', 'player_move', 'policy_phase', 'outcome']
valid_sessions = []

for df_tidy, fname in zip(sessions, csv_files):
    scored_rounds = df_tidy.loc[(df_tidy['task_phase'] == 'rps') & (df_tidy['practice'] == False)]
    if scored_rounds.shape[0] < 100:
        print(f'Skipping {fname}')
        continue
    session = scored_rounds[kept_columns]
    
    valid_sessions.append(session)

df_tidy = pd.concat(valid_sessions, ignore_index=True)

# Cast to appropriate dtypes
column_dtypes = {
    'round_index': int,
    'policy_phase': int,
}

df_tidy = df_tidy.astype(column_dtypes)
df_tidy['won'] = df_tidy['outcome'].map({'win': True, 'loss': False})
df_tidy = df_tidy.drop('outcome', axis=1)

display(df_tidy.head())
display(df_tidy.describe())

In [ ]:
# Remove any participants with unrealistic data
MAX_POLICY_ACC = 0.95

df = df_tidy.copy()

mean_rt = df.groupby('random_id')['rt'].mean()
extreme_rt_ids = mean_rt[(mean_rt < 150)].index
df = df[~df['random_id'].isin(extreme_rt_ids)]
print(f"Removed {len(extreme_rt_ids)} participants with mean rt < 150")

move_counts = df.groupby('random_id')['player_move'].nunique()
low_move_var_ids = move_counts[move_counts < 3].index
df = df[~df['random_id'].isin(low_move_var_ids)]
print(f"Removed {len(low_move_var_ids)} participants using fewer than 3 moves")

mean_acc = df.loc[df['policy_phase'] > 0].groupby('random_id')['won'].mean()

high_acc_ids = mean_acc[mean_acc > MAX_POLICY_ACC].index
df = df[~df['random_id'].isin(high_acc_ids)]
print(f"Removed {len(high_acc_ids)} participants with greater than {MAX_POLICY_ACC} mean accuracy")

In [ ]:
df.to_csv(f'{DATA_DIR}/clean.csv', index=False)

In [ ]:
# Filter the dataframe by rounds during which the participant
# passed a threshold minimum mean accuracy
# (i.e. had "learned" the hidden policy)

BURN_ROUNDS = 50
ACC_THRESHOLD = 0.7

df_latter = df.loc[df['round_index'] > BURN_ROUNDS].copy()
latter_acc = df_latter.loc[df['policy_phase'] > 0].groupby('random_id')['won'].mean()
latter_acc_hi = latter_acc[latter_acc > ACC_THRESHOLD]
hiacc_ids = latter_acc_hi.index
df_hiacc = df.loc[df['random_id'].isin(hiacc_ids)].copy()

print(df_hiacc['random_id'].nunique())

df_hiacc.to_csv(DATA_DIR / 'hiacc.csv', index=False)

In [ ]:
chunk_lengths_hi = df_hiacc.groupby('random_id')['chunk_length'].first()
rt_hi = df_hiacc.groupby('random_id')['rt'].mean()

ax = sns.regplot(x=chunk_lengths_hi, y=rt_hi, order=2)
ax.set_xticks(range(2, 6))

In [ ]:
cl_vs_rt = smf.ols('rt ~ chunk_length + I(chunk_length**2)', data={
    'rt': rt_hi,
    'chunk_length': chunk_lengths_hi,
}).fit()

cl_vs_rt.summary()

In [ ]:
acc_hi = df_hiacc.groupby('random_id')['won'].mean()
sns.regplot(x=chunk_lengths_hi, y=acc_hi)

In [ ]:
cl_vs_acc = smf.ols('acc_hi ~ chunk_length', data={
    'acc_hi': acc_hi,
    'chunk_length': chunk_lengths_hi,
}).fit()

cl_vs_acc.summary()